# P0 · Second backbone: DINOv2-B (exploratory)

The referee asked whether the codebook-vs-coreset result depends on DINOv2-S. This reruns the main comparison
with **DINOv2-B** (768-d patch tokens), same protocol: 448 px, M = K ∈ {64, 256, 1,024}, 3 seeds, codebook vs.
greedy coreset (plus the full bank on MVTec AD and VisA).

**One dataset per Colab session:** set `DATASET` below to `"mvtec"`, then `"visa"`, then `"3cad"`.
**Runtime (T4):** MVTec about 2–3 h, VisA about 4–5 h, 3CAD about 5–6 h. Finished runs are saved to Drive;
after a disconnect, Run all again.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Data

In [ ]:
DATASET = "mvtec"     # "mvtec", then "visa", then "3cad" (one per session)

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then Run all.")
import os, glob, shutil, subprocess
from bovw import ad_data

def get_mvtec():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/mvtec_anomaly_detection.tar.xz", "/content/mvtec"
    if not os.path.isfile(tar):
        raise SystemExit("Put mvtec_anomaly_detection.tar.xz in MyDrive/bovw-forensics/data/ (see notebook 02).")
    if not os.path.isdir(f"{local}/bottle"):
        os.makedirs(local, exist_ok=True)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return local

def get_visa():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/VisA_20220922.tar", "/content/visa"
    if not os.path.isfile(tar) or os.path.getsize(tar) < 1e9:
        ad_data.ensure_archive(tar, url=ad_data.VISA_URL, sha256=None)
    if not os.path.isdir(local):
        os.makedirs(local)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return ad_data.find_visa_root(local)

def get_3cad():
    local, archive = "/content/3cad", "/content/3cad_archive"
    if not os.path.isdir(local):
        copy = glob.glob("/content/drive/MyDrive/bovw-forensics/data/3CAD*")
        if not copy:
            raise SystemExit("Put the 3CAD archive in MyDrive/bovw-forensics/data/ (see notebook 06).")
        shutil.copy(copy[0], archive)
        os.makedirs(local)
        magic = open(archive, "rb").read(4)
        fmt = "zip" if magic[:2] == b"PK" else ("gztar" if magic[:2] == b"\x1f\x8b" else "tar")
        shutil.unpack_archive(archive, local, fmt)
        os.remove(archive)
    return ad_data.find_root(local, "Copper_Stator")

ROOT = {"mvtec": get_mvtec, "visa": get_visa, "3cad": get_3cad}[DATASET]()
print(DATASET, ROOT)

## 3 · Run

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from bovw.sweep import load_config
from bovw import anomaly, stats

cfg = load_config(f"configs/p0_dinov2b_{DATASET}.yaml")
cfg["data"]["root"] = ROOT
df = anomaly.run(cfg)

## 4 · Codebook vs. coreset with DINOv2-B (paired over categories)

In [ ]:
RES = cfg["results_dir"]
pb = stats.compare({DATASET: df}, "codebook_dist", "coreset_knn", ks=(64, 256, 1024))
pb.to_csv(f"{RES}/p0_dinov2b_{DATASET}_paired.csv", index=False)
print(anomaly.followup_table(df).round(3).to_string())
pb.drop(columns=["p_sign", "p_wilcoxon"]).round(3)